# Milestone 2 — Conditional Generation & Tuning

**Model:** Stable Diffusion v1.5 · **Encoder:** CLIP ViT-L/14 · **Metric:** CLIP score

1. **Integrate** the text encoder with the diffusion model, explicitly, through `prompt_embeds`.
2. **Baseline** conditional generation on a fixed prompt set.
3. **Tune** classifier-free guidance, the sampler, and the noise schedule.
4. **Record** the first set of generated images.

No weights are trained. Every one of those is satisfied at inference time, which is what makes the
whole thing fit in under half an hour on one consumer GPU.

---
## 1. Setup

Artefacts are written to `/kaggle/working/outputs/`. The caption manifest and the 512px reference
photos are read from the attached Dataset and copied to `/kaggle/working/data/` first, because
`/kaggle/input/` is read-only. Off Kaggle both fall back to `./data/` and `./outputs/`.

In [ ]:
import json, math, os, shutil, time, warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import diffusers
import transformers
from diffusers import (DDIMScheduler, DDPMScheduler, DPMSolverMultistepScheduler,
                       EulerAncestralDiscreteScheduler, EulerDiscreteScheduler,
                       LMSDiscreteScheduler, PNDMScheduler, StableDiffusionPipeline,
                       UniPCMultistepScheduler)
from transformers import CLIPImageProcessor, CLIPModel, CLIPTextModel, CLIPTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()


def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        data, out = Path("/kaggle/working/data"), Path("/kaggle/working/outputs")
        if not (data / "processed").is_dir():
            print(f"copying {found[0].parent} -> {data / 'processed'} ...")
            shutil.copytree(found[0].parent, data / "processed")
        return data, out
    return Path("data"), Path("outputs")


DATA_DIR, OUT_DIR = resolve_paths()
PROCESSED = DATA_DIR / "processed"
SWEEP_DIR, PLOTS_DIR, TABLES_DIR = OUT_DIR / "sweep", OUT_DIR / "plots", OUT_DIR / "tables"
for d in (SWEEP_DIR, PLOTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

print(f"torch {torch.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
print(f"device  : {DEVICE}")
if DEVICE.type == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"gpu     : {torch.cuda.get_device_name(0)}  {gb:.1f} GB  sm_{major}{minor}")
    print(f"kernels : {'yes' if f'sm_{major}{minor}' in torch.cuda.get_arch_list() else 'NO -- wrong wheel'}")
print(f"data    : {PROCESSED}")
print(f"outputs : {OUT_DIR}")

### Settings

Every knob the run touches. `BATCH_SIZE` is the one to raise on a bigger card: 4 suits an 8 GB
laptop GPU, a 16 GB T4 takes 8 comfortably.

The samplers each carry **their own step count**. An equal step count across samplers would measure
the step count rather than the sampler, since DPMSolver++ reaches at 20 steps what PNDM needs 50 for.

In [ ]:
SEED = 42
N_PROMPTS = 10
PROMPT_MIN_WORDS, PROMPT_MAX_WORDS = 8, 18

IMAGE_SIZE = 512
BATCH_SIZE = 4
GUIDANCE = 7.5
GUIDANCE_RESCALE = 0.7          # corrects the oversaturation CFG above ~12 causes
BASELINE_STEPS = 50

CFG_GRID = (1.0, 4.0, 7.5, 11.0, 15.0)
SAMPLERS = {"pndm": 50, "ddim": 50, "lms": 30, "euler": 30, "euler_a": 30,
            "dpmpp": 20, "unipc": 20}

TIMESTEP_SPACINGS = ("leading", "trailing", "linspace")
ZERO_SNR_OPTIONS = (False, True)
KARRAS_OPTIONS = (False, True)
SCHEDULE_SAMPLER = "dpmpp"
BASE_SPACING = "leading"        # SD 1.5 ships no value for this; see section 8

DDPM_STEPS = 1000
DDPM_N_PROMPTS = 2

CLIP_JOINT_MODEL = "openai/clip-vit-base-patch32"
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"
CLIP_MAX_TOKENS = 77
SD_REPOS = ["stable-diffusion-v1-5/stable-diffusion-v1-5",
            "sd-legacy/stable-diffusion-v1-5",
            "CompVis/stable-diffusion-v1-4"]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"seed {SEED} | {N_PROMPTS} prompts | batch {BATCH_SIZE} | {SD_DTYPE}")

---
## 2. The fixed prompt set

A sweep is only a sweep if the prompts never move. Every configuration below renders **the same ten
captions from the same ten latents**, so a difference in CLIP score is a difference in the knob and
nothing else.

The captions come from the curated manifest rather than being written by hand, so they are real
captions the encoders have already been measured on. The 8–18 word band matters: a three-word
caption underdetermines the image, so everything scores well and the sweep separates nothing.

In [ ]:
manifest = pd.read_parquet(PROCESSED / "pairs.parquet")

eligible = manifest[manifest["n_words"].between(PROMPT_MIN_WORDS, PROMPT_MAX_WORDS)]
if len(eligible) < N_PROMPTS:
    raise ValueError(f"{N_PROMPTS} prompts requested but only {len(eligible)} captions fall in "
                     f"{PROMPT_MIN_WORDS}-{PROMPT_MAX_WORDS} words")

picks = eligible.sample(n=N_PROMPTS, random_state=SEED).sort_values("pair_idx").reset_index(drop=True)
PROMPTS = picks["caption"].tolist()
REAL_IMAGES = [PROCESSED / "images_512" / Path(p).name for p in picks["image_512_path"]]

print(f"{len(manifest):,} curated pairs -> {len(PROMPTS)} fixed prompts\n")
for i, (caption, n_words) in enumerate(zip(PROMPTS, picks["n_words"])):
    print(f"  {i:>2}  [{n_words:>2}w]  {caption}")
print(f"\nreference photos found: {sum(p.is_file() for p in REAL_IMAGES)}/{len(REAL_IMAGES)}")

### The starting noise

Fixing the seed is **not enough**. Ancestral and SDE samplers draw extra noise inside the denoising
loop, and different samplers consume the RNG stream differently, so two configurations sharing a
seed do not share a starting point. The only way to hold it still is to generate the noise once and
pass the tensor in.

It is drawn on a **CPU** generator, because `torch.Generator(device="cuda")` is a different RNG
stream — a CUDA-seeded run does not reproduce on another machine or another GPU.

Row *i* is prompt *i*'s noise, in every configuration.

In [ ]:
def make_latents(n, height=IMAGE_SIZE, width=IMAGE_SIZE, seed=SEED, channels=4, vae_scale=8):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    return torch.randn((n, channels, height // vae_scale, width // vae_scale), generator=generator)


# One generator per prompt, so the batch size cannot change a result. Pre-generated
# latents pin only where a run starts; ancestral samplers and DDPM keep drawing inside
# the loop, and without this they would draw from the global RNG.
def prompt_generators(n, seed=SEED):
    return [torch.Generator(device="cpu").manual_seed(seed + i) for i in range(n)]


LATENTS = make_latents(len(PROMPTS))
print(f"latents      : {tuple(LATENTS.shape)}  {LATENTS.dtype}  on {LATENTS.device}")
print(f"reproducible : {torch.equal(LATENTS, make_latents(len(PROMPTS)))}")
print(f"generators   : {len(prompt_generators(len(PROMPTS)))}, seeds {SEED}..{SEED + len(PROMPTS) - 1}")

---
## 3. Integrating the text encoder — deliverable 1

Normally you pass `prompt="a dog"` and the pipeline runs CLIP for you. **Integrating the encoder
explicitly means doing that half ourselves**: load the text encoder, produce the `(N, 77, 768)`
conditioning tensor, and hand it to the UNet. There is no `prompt=` anywhere after this, except in
the comparison below that exists precisely to check the two paths agree.

Per-token states, not the pooled vector — the UNet cross-attends to each token position, which is
how "a *red* bus" colours the bus instead of tinting the whole frame.

**The encoder is called without an attention mask.** CLIP's text tower is causal, so a mask leaves
every real token bit-identical and moves only the padded positions — which is why the mistake
survives a casual check, and why it still changes the image, since the UNet cross-attends to all 77
positions. Getting this wrong costs about 0.11 of CLIP score with nothing raising an error.

The encoder is freed before Stable Diffusion loads. The embeddings are about a megabyte; the
encoder is 1.4 GB.

In [ ]:
@torch.no_grad()
def encode_prompts(prompts, tokenizer, text_encoder, device=DEVICE):
    tokens = tokenizer(prompts, padding="max_length", max_length=CLIP_MAX_TOKENS,
                       truncation=True, return_tensors="pt").to(device)
    wants_mask = getattr(text_encoder.config, "use_attention_mask", False)
    hidden = text_encoder(input_ids=tokens["input_ids"],
                          attention_mask=tokens["attention_mask"] if wants_mask else None)
    return hidden.last_hidden_state


def max_abs_difference(a, b):
    return float((a.float() - b.float()).abs().max())


sd_tokenizer = CLIPTokenizer.from_pretrained(CLIP_SD_MODEL)
sd_text_encoder = CLIPTextModel.from_pretrained(CLIP_SD_MODEL).to(DEVICE, torch.float32).eval()

POS = encode_prompts(PROMPTS, sd_tokenizer, sd_text_encoder).cpu()
NEG = encode_prompts([""] * len(PROMPTS), sd_tokenizer, sd_text_encoder).cpu()

print(f"encoder       : {CLIP_SD_MODEL}")
print(f"prompt_embeds : {tuple(POS.shape)}  {POS.dtype}")
print(f"negative      : {tuple(NEG.shape)}  (the empty string, one row per prompt)")
print(f"size          : {POS.numel() * POS.element_size() / 1024**2:.1f} MB")

del sd_text_encoder
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

---
## 4. Loading Stable Diffusion 1.5

The original `runwayml` repository was withdrawn, so mirrors are tried in order.

The scheduler's **original configuration is captured here**, before anything touches it. Every
sampler swap later rebuilds from this dict rather than from the live scheduler: swapping twice from
the live one compounds the first swap's overrides into the second, and `beta_schedule` silently
drifts away from the `scaled_linear` SD 1.5 was trained on.

In [ ]:
def load_sd(repos=SD_REPOS, device=DEVICE, dtype=SD_DTYPE):
    errors = {}
    for repo in repos:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(
                repo, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False)
        except Exception as exc:
            errors[repo] = f"{type(exc).__name__}: {exc}"
            continue
        pipe = pipe.to(device)
        pipe.set_progress_bar_config(disable=True)
        if device.type == "cuda":
            pipe.enable_attention_slicing()
            if hasattr(pipe.vae, "enable_slicing"):
                pipe.vae.enable_slicing()
        return pipe, repo, dict(pipe.scheduler.config)
    raise RuntimeError("no Stable Diffusion repo could be loaded: " + repr(errors))


def vram():
    if DEVICE.type != "cuda":
        return {}
    gb = 1024 ** 3
    return {"allocated_gb": round(torch.cuda.memory_allocated() / gb, 2),
            "reserved_gb": round(torch.cuda.memory_reserved() / gb, 2),
            "peak_gb": round(torch.cuda.max_memory_allocated() / gb, 2)}


t0 = time.time()
pipe, SD_REPO, BASE_SCHEDULER_CONFIG = load_sd()
print(f"loaded        : {SD_REPO}  in {time.time() - t0:.1f}s")
print(f"scheduler     : {type(pipe.scheduler).__name__}")
print(f"beta_schedule : {BASE_SCHEDULER_CONFIG['beta_schedule']}")
print(f"train steps   : {BASE_SCHEDULER_CONFIG['num_train_timesteps']}")
print(f"vram          : {vram()}")

### Does the hand-wired path match the built-in one?

The check that the integration is real. The pipeline encodes the same prompt its own way, and the
two tensors are compared, then the same latent is rendered through both paths.

Expect *near*-identical, not bit-identical: this encodes in float32 and the pipeline runs float16, so
a difference around `1e-2` on a tensor whose values reach ~33 is the precision gap. A difference of
zero would mean we accidentally called the pipeline's own encoder; a large one means the tensors are
not what the UNet expects.

In [ ]:
with torch.no_grad():
    pipe_pos, pipe_neg = pipe.encode_prompt(
        prompt=PROMPTS[:1], device=DEVICE, num_images_per_prompt=1,
        do_classifier_free_guidance=True, negative_prompt=[""])

delta_pos = max_abs_difference(POS[:1].to(DEVICE), pipe_pos)
delta_neg = max_abs_difference(NEG[:1].to(DEVICE), pipe_neg)

print(f"ours   : {tuple(POS[:1].shape)}  {POS.dtype}")
print(f"theirs : {tuple(pipe_pos.shape)}  {pipe_pos.dtype}")
print(f"max |difference|  positive {delta_pos:.3e}   negative {delta_neg:.3e}")
print(f"embedding scale   {float(POS.abs().max()):.2f}   -> relative {delta_pos / float(POS.abs().max()):.2e}")

---
## 5. Swapping samplers without losing SD 1.5's training

`DDIMScheduler()` would construct a scheduler with *library* defaults, which are not SD 1.5's —
`beta_schedule` becomes `linear` where SD 1.5 trained on `scaled_linear`, and every image comes out
wrong for a reason nothing reports. The swap goes through `from_config`.

The noise-schedule options are not universal either. `EulerAncestralDiscreteScheduler` has no
`use_karras_sigmas`; `PNDMScheduler` has no `rescale_betas_zero_snr`. Passing one anyway is a
`TypeError` partway through a run, so the accepted set is read from each class's signature rather
than hardcoded.

In [ ]:
import inspect

SCHEDULER_CLASSES = {
    "pndm": PNDMScheduler, "ddim": DDIMScheduler, "lms": LMSDiscreteScheduler,
    "euler": EulerDiscreteScheduler, "euler_a": EulerAncestralDiscreteScheduler,
    "dpmpp": DPMSolverMultistepScheduler, "unipc": UniPCMultistepScheduler,
    "ddpm": DDPMScheduler,          # the 1000-step illustration, not a sweep arm
}
SCHEDULER_DEFAULTS = {"dpmpp": {"algorithm_type": "dpmsolver++"}}


def scheduler_kwargs(name, **overrides):
    cls = SCHEDULER_CLASSES[name]
    merged = {**SCHEDULER_DEFAULTS.get(name, {}), **overrides}
    params = inspect.signature(cls.__init__).parameters
    return {k: v for k, v in merged.items() if k in params}


def set_scheduler(pipe, name, base_config=None, **overrides):
    base = dict(BASE_SCHEDULER_CONFIG if base_config is None else base_config)
    pipe.scheduler = SCHEDULER_CLASSES[name].from_config(base, **scheduler_kwargs(name, **overrides))
    return pipe


for name in SAMPLERS:
    set_scheduler(pipe, name)
    print(f"  {name:<8} {type(pipe.scheduler).__name__:<34} beta={pipe.scheduler.config['beta_schedule']}")
print("\nkarras accepted by:", [n for n in SAMPLERS if "use_karras_sigmas" in scheduler_kwargs(n, use_karras_sigmas=True)])
print("zero-snr accepted by:", [n for n in SAMPLERS if "rescale_betas_zero_snr" in scheduler_kwargs(n, rescale_betas_zero_snr=True)])

---
## 6. One configuration, and the call that renders it

`GenConfig` is frozen and its `scheduler_kwargs` is a tuple of pairs rather than a dict, so a
configuration is hashable and its `label` is a stable identifier — used as the directory name for
that configuration's images.

Latents and generators are sliced alongside the embeddings, so prompt *i* keeps its own noise
wherever the batch boundary falls and changing the batch size cannot change a result.

In [ ]:
LABEL_FLAGS = {"use_karras_sigmas": "karras", "rescale_betas_zero_snr": "zerosnr"}


@dataclass(frozen=True)
class GenConfig:
    sampler: str
    steps: int
    guidance: float
    guidance_rescale: float = 0.0
    scheduler_kwargs: tuple = field(default=())

    @property
    def label(self):
        parts = [self.sampler, f"s{self.steps}", f"g{self.guidance:g}"]
        if self.guidance_rescale:
            parts.append(f"r{self.guidance_rescale:g}")
        for key, value in self.scheduler_kwargs:
            if value is True:
                parts.append(LABEL_FLAGS.get(key, key))
            elif value is not False and value is not None:
                parts.append(str(value))
        return "_".join(parts)


def batches(n, size):
    return [slice(i, min(i + size, n)) for i in range(0, n, size)]


def generate(config, prompt_embeds=None, negative_embeds=None, latents=None, batch_size=BATCH_SIZE):
    prompt_embeds = POS if prompt_embeds is None else prompt_embeds
    negative_embeds = NEG if negative_embeds is None else negative_embeds
    latents = LATENTS if latents is None else latents

    set_scheduler(pipe, config.sampler, **dict(config.scheduler_kwargs))
    generators = prompt_generators(len(prompt_embeds))

    images = []
    for part in batches(len(prompt_embeds), batch_size):
        out = pipe(
            prompt_embeds=prompt_embeds[part].to(DEVICE, pipe.dtype),
            negative_prompt_embeds=negative_embeds[part].to(DEVICE, pipe.dtype),
            latents=latents[part].to(DEVICE, pipe.dtype),
            generator=generators[part],
            num_inference_steps=config.steps,
            guidance_scale=config.guidance,
            guidance_rescale=config.guidance_rescale,
        )
        images.extend(out.images)
    return images


print(GenConfig("dpmpp", 20, 7.5).label)
print(GenConfig("dpmpp", 20, 15.0, 0.7, (("timestep_spacing", "trailing"), ("use_karras_sigmas", True))).label)
print("batches(10, 4) ->", batches(10, 4))

---
## 7. Scoring, and the baseline — deliverable 2

The metric is a cosine in CLIP ViT-B/32's joint text-image space. It is a **different** CLIP from
the ViT-L/14 that conditions the generator, so the judge is not the contestant's own encoder.

For reference, the **real photographs** these captions describe score around 0.31. That is the
number to read the generated scores against, not 1.0. A text-image cosine above roughly 0.30 means
the image matches the caption; 0.1 means it does not.

The baseline is PNDM at CFG 7.5 — SD 1.5's own defaults, and a point that also appears in the
guidance grid below, so "improvement" later subtracts a configuration that really was measured.

In [ ]:
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_JOINT_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_JOINT_MODEL).to(DEVICE, torch.float32).eval()
clip_processor = CLIPImageProcessor.from_pretrained(CLIP_JOINT_MODEL)


def pooled(out):
    return out.pooler_output if hasattr(out, "pooler_output") else out


@torch.no_grad()
def embed_images(images, batch_size=32):
    out = []
    for i in range(0, len(images), batch_size):
        batch = [(img if isinstance(img, Image.Image) else Image.open(img)).convert("RGB")
                 for img in images[i:i + batch_size]]
        px = clip_processor(images=batch, return_tensors="pt").to(DEVICE)
        out.append(pooled(clip_model.get_image_features(**px)).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def embed_texts(texts):
    toks = clip_tokenizer(texts, padding="max_length", max_length=CLIP_MAX_TOKENS,
                          truncation=True, return_tensors="pt").to(DEVICE)
    return F.normalize(pooled(clip_model.get_text_features(**toks)).float().cpu(), dim=-1)


TEXT_EMB = embed_texts(PROMPTS)
REAL_EMB = embed_images(REAL_IMAGES)
REFERENCE = float((TEXT_EMB * REAL_EMB).sum(-1).mean())


# CLIP(text, generated), one cosine per prompt.
def clip_score(images):
    return (TEXT_EMB * embed_images(images)).sum(-1).numpy()


print(f"CLIP(text, real photo) = {REFERENCE:.4f}   <- the reference every score is read against")

In [ ]:
BASELINE = GenConfig("pndm", SAMPLERS["pndm"], GUIDANCE,
                     scheduler_kwargs=(("timestep_spacing", BASE_SPACING),))

t0 = time.time()
baseline_images = generate(BASELINE)
baseline_seconds = time.time() - t0
baseline_scores = clip_score(baseline_images)

print(f"config  : {BASELINE.label}")
print(f"time    : {baseline_seconds:.1f}s for {len(PROMPTS)} images "
      f"({baseline_seconds / len(PROMPTS):.2f}s each, "
      f"{baseline_seconds / len(PROMPTS) / BASELINE.steps:.3f}s per step)")
print(f"CLIP    : {baseline_scores.mean():.4f} +/- {baseline_scores.std():.4f}   (real photos {REFERENCE:.4f})")
print(f"vram    : {vram()}")

### Recording the images — deliverable 7

Fifty PNGs in a folder say nothing; one grid per configuration says everything. Labels are
slugified because captions contain characters a filesystem will not accept.

In [ ]:
import re

UNSAFE = re.compile(r"[^A-Za-z0-9._-]+")


def slugify(text, max_length=60):
    return UNSAFE.sub("_", str(text)).strip("_")[:max_length] or "untitled"


def image_grid(images, rows=None, cols=None, pad=4, background=(255, 255, 255)):
    if not images:
        raise ValueError("image_grid needs at least one image")
    if rows is None and cols is None:
        cols = math.ceil(math.sqrt(len(images)))
    if cols is None:
        cols = math.ceil(len(images) / rows)
    if rows is None:
        rows = math.ceil(len(images) / cols)
    w, h = images[0].size
    canvas = Image.new("RGB", (cols * w + (cols + 1) * pad, rows * h + (rows + 1) * pad), background)
    for i, img in enumerate(images):
        r, c = divmod(i, cols)
        canvas.paste(img, (pad + c * (w + pad), pad + r * (h + pad)))
    return canvas


def save_run(images, labels, name, out_dir=None):
    directory = (SWEEP_DIR if out_dir is None else out_dir) / name
    directory.mkdir(parents=True, exist_ok=True)
    for i, (img, label) in enumerate(zip(images, labels)):
        img.save(directory / f"{i:02d}_{slugify(label)}.png")
    image_grid(images, rows=2, cols=5).save(directory / "grid.png")
    return directory


print("wrote", save_run(baseline_images, PROMPTS, "baseline"))

fig, axes = plt.subplots(2, 5, figsize=(15, 6.8))
for ax, img, caption, score in zip(axes.ravel(), baseline_images, PROMPTS, baseline_scores):
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{score:.3f}  {caption[:34]}...", fontsize=7)
fig.suptitle(f"Baseline — {BASELINE.label} — CLIP(txt,gen) {baseline_scores.mean():.4f}", fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_baseline_grid.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. The sweep

`run_sweep` writes the results table **after every configuration**, not at the end. A configuration
can fail with an out-of-memory error partway through a long run, and losing the twenty that already
succeeded to the twenty-first is the expensive mistake. A failure becomes a row with
`status="error"` and the run carries on.

In [ ]:
COLUMNS = ["label", "sampler", "steps", "guidance", "guidance_rescale", "scheduler_kwargs",
           "clip_score_mean", "clip_score_std", "seconds", "status", "error"]


def run_sweep(configs, name, save_images=True):
    table_path = SWEEP_DIR / f"{name}.csv"
    rows = []
    for config in tqdm(configs, desc=name):
        row = {"label": config.label, "sampler": config.sampler, "steps": config.steps,
               "guidance": config.guidance, "guidance_rescale": config.guidance_rescale,
               "scheduler_kwargs": str(dict(config.scheduler_kwargs)),
               "clip_score_mean": np.nan, "clip_score_std": np.nan,
               "status": "ok", "error": ""}
        started = time.time()
        try:
            images = generate(config)
            scores = clip_score(images)
            row["clip_score_mean"] = float(scores.mean())
            row["clip_score_std"] = float(scores.std())
            if save_images:
                save_run(images, PROMPTS, config.label)
        except Exception as exc:
            row["status"], row["error"] = "error", f"{type(exc).__name__}: {exc}"
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        row["seconds"] = round(time.time() - started, 2)
        rows.append(row)
        pd.DataFrame(rows)[COLUMNS].to_csv(table_path, index=False)
    return pd.DataFrame(rows)[COLUMNS]

### Classifier-free guidance — deliverable 3

CFG is the dial between "ignore the prompt" and "obey the prompt at any cost". At 1.0 there is no
guidance at all; as it rises the image tracks the caption more closely but saturates and loses
detail. Above about 12 the oversaturation needs `guidance_rescale` to correct, so the top of the
grid carries it — which does mean that point moves two knobs, and the plot says so.

In [ ]:
PINNED = (("timestep_spacing", BASE_SPACING),)

guidance_grid = [GenConfig("pndm", SAMPLERS["pndm"], g,
                           guidance_rescale=GUIDANCE_RESCALE if g > 12 else 0.0,
                           scheduler_kwargs=PINNED)
                 for g in CFG_GRID]

guidance_results = run_sweep(guidance_grid, "guidance_results")
display(guidance_results[["label", "guidance", "guidance_rescale", "clip_score_mean",
                          "clip_score_std", "seconds", "status"]])

In [ ]:
ok = guidance_results[guidance_results["status"] == "ok"]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.errorbar(ok["guidance"], ok["clip_score_mean"], yerr=ok["clip_score_std"],
            marker="o", capsize=4, label="generated")
ax.axhline(REFERENCE, ls="--", c="grey", label=f"real photos ({REFERENCE:.3f})")
ax.annotate("rescale 0.7 on", (CFG_GRID[-1], ok["clip_score_mean"].iloc[-1]),
            textcoords="offset points", xytext=(-70, 12), fontsize=8, color="crimson")
ax.set_xlabel("guidance_scale"); ax.set_ylabel("CLIP(text, generated)")
ax.set_title("Prompt adherence against classifier-free guidance")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_guidance_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

### Samplers — deliverable 4

Seven samplers, each at its own step count, all from the same latents.

`timestep_spacing` is **pinned** here. SD 1.5's shipped scheduler configuration contains no value
for it, so each scheduler class would otherwise fall back to its own default — PNDM and DDIM to
`leading`, the other five to `linspace` — and the comparison would quietly vary two things at once.

In [ ]:
sampler_grid = [GenConfig(name, steps, GUIDANCE, scheduler_kwargs=PINNED)
                for name, steps in SAMPLERS.items()]

sampler_results = run_sweep(sampler_grid, "sampler_results")

table = sampler_results.copy()
table["s_per_image"] = (table["seconds"] / len(PROMPTS)).round(2)
display(table[["sampler", "steps", "clip_score_mean", "clip_score_std", "seconds",
               "s_per_image", "status"]].sort_values("clip_score_mean", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
ok = sampler_results[sampler_results["status"] == "ok"]
ax.scatter(ok["seconds"], ok["clip_score_mean"], s=60)
for _, row in ok.iterrows():
    ax.annotate(f"{row['sampler']} ({row['steps']})", (row["seconds"], row["clip_score_mean"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel(f"seconds for {len(PROMPTS)} images"); ax.set_ylabel("CLIP(text, generated)")
ax.set_title("Quality against cost, one point per sampler")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_sampler_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

### Noise schedule — deliverable 5

These are scheduler *configuration*, not call arguments — they change how the noise ramp is shaped
and which timesteps get visited, so they have to be baked in when the scheduler is constructed.

| Knob | Options | Effect |
|---|---|---|
| `timestep_spacing` | `leading`, `trailing`, `linspace` | which timesteps get visited |
| `rescale_betas_zero_snr` | True / False | fixes "cannot generate a truly dark image" |
| `use_karras_sigmas` | True / False | smoother spacing, helps most at low step counts |

Twelve combinations on one sampler at one guidance scale. CLIP score is a blunt instrument for the
zero-SNR fix in particular: it measures caption adherence, and "can render a genuinely black image"
is something it barely sees. Read the grids as well as the table.

In [ ]:
schedule_grid = [
    GenConfig(SCHEDULE_SAMPLER, SAMPLERS[SCHEDULE_SAMPLER], GUIDANCE, scheduler_kwargs=(
        ("timestep_spacing", spacing),
        ("rescale_betas_zero_snr", zero_snr),
        ("use_karras_sigmas", karras)))
    for spacing in TIMESTEP_SPACINGS
    for zero_snr in ZERO_SNR_OPTIONS
    for karras in KARRAS_OPTIONS
]
print(f"{len(schedule_grid)} combinations on {SCHEDULE_SAMPLER} at CFG {GUIDANCE}")

schedule_results = run_sweep(schedule_grid, "schedule_results")
display(schedule_results[["scheduler_kwargs", "clip_score_mean", "clip_score_std",
                          "seconds", "status"]].sort_values("clip_score_mean", ascending=False))

### DDPM — 1000 steps against 20 — deliverable 6

DDPM is the original formulation and it is **not a sweep arm**: at ~1000 steps it costs roughly a
hundred times what DPMSolver++ costs, for a worse image. Running it once on two prompts is the
honest way to show that — the story is "the samplers above are what made diffusion practical", and
it is worth a few minutes to show rather than assert.

In [ ]:
k = DDPM_N_PROMPTS
slow = GenConfig("ddpm", DDPM_STEPS, GUIDANCE)
fast = GenConfig("dpmpp", SAMPLERS["dpmpp"], GUIDANCE)

panels, timings = {}, {}
for config in (slow, fast):
    t0 = time.time()
    panels[config.label] = generate(config, POS[:k], NEG[:k], LATENTS[:k], batch_size=k)
    timings[config.label] = time.time() - t0
    print(f"  {config.label:<24} {timings[config.label]:7.1f}s  "
          f"({timings[config.label] / k:.1f}s per image)")

print(f"\nspeed-up: {timings[slow.label] / timings[fast.label]:.0f}x for "
      f"{slow.steps // fast.steps}x fewer steps")

fig, axes = plt.subplots(2, k, figsize=(4.2 * k, 8.8), squeeze=False)
for row, (label, images) in enumerate(panels.items()):
    for col, img in enumerate(images):
        axes[row, col].imshow(img); axes[row, col].axis("off")
    axes[row, 0].set_title(f"{label} — {timings[label]:.0f}s", fontsize=9, loc="left")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_ddpm_vs_dpmpp.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Results — deliverable 7

Everything above in one table and one picture. PNDM at CFG 7.5 is a member of both the guidance and
the sampler grid — the same configuration, rendered twice — so it is de-duplicated before ranking,
otherwise one configuration would occupy two places in the same table.

In [ ]:
all_results = pd.concat([
    guidance_results.assign(axis="guidance"),
    sampler_results.assign(axis="sampler"),
    schedule_results.assign(axis="schedule"),
], ignore_index=True)

duplicates = int(all_results["label"].duplicated().sum())
all_results = all_results.drop_duplicates(subset="label", keep="first").reset_index(drop=True)

ranked = (all_results[all_results["status"] == "ok"]
          .sort_values("clip_score_mean", ascending=False).reset_index(drop=True))
if ranked.empty:
    raise RuntimeError("every configuration failed -- read the 'error' column in "
                       f"{SWEEP_DIR / 'all_results.csv'}")
best = ranked.iloc[0]

print(f"configurations : {len(all_results)}  "
      f"({(all_results['status'] == 'error').sum()} failed, {duplicates} duplicates dropped)")
print(f"total GPU time : {all_results['seconds'].sum() / 60:.1f} min\n")
print(f"baseline       : {BASELINE.label:<36}{baseline_scores.mean():.4f}")
print(f"best           : {best['label']:<36}{best['clip_score_mean']:.4f}")
print(f"real photos    : {'(reference)':<36}{REFERENCE:.4f}")
print(f"improvement    : {best['clip_score_mean'] - baseline_scores.mean():+.4f}\n")
display(ranked.head(10)[["axis", "label", "clip_score_mean", "clip_score_std", "seconds"]])

In [ ]:
best_config = next(c for c in (*guidance_grid, *sampler_grid, *schedule_grid)
                   if c.label == best["label"])
best_images = generate(best_config)

from PIL import ImageDraw, ImageFont

BAND = 20
rows = {f"baseline  {BASELINE.label}  ({baseline_scores.mean():.4f})": baseline_images,
        f"best      {best_config.label}  ({best['clip_score_mean']:.4f})": best_images}

strips = {label: image_grid(imgs, rows=1, cols=len(imgs)) for label, imgs in rows.items()}
width = max(s.width for s in strips.values())
canvas = Image.new("RGB", (width, sum(s.height + BAND for s in strips.values())), (255, 255, 255))
draw, font, y = ImageDraw.Draw(canvas), ImageFont.load_default(), 0
for label, strip in strips.items():
    draw.text((6, y + 4), label, fill=(0, 0, 0), font=font)
    canvas.paste(strip, (0, y + BAND))
    y += BAND + strip.height

canvas.save(SWEEP_DIR / "baseline_vs_best.png")
print(SWEEP_DIR / "baseline_vs_best.png")
canvas

In [ ]:
all_results.to_csv(SWEEP_DIR / "all_results.csv", index=False)

summary = {
    "prompts": PROMPTS,
    "prompt_pair_idx": picks["pair_idx"].tolist(),
    "seed": SEED,
    "repo": SD_REPO,
    "integration": {"max_abs_embedding_difference": delta_pos,
                    "note": "float32 encoder against float16 pipeline"},
    "baseline": {"config": BASELINE.label,
                 "clip_text_generated": float(baseline_scores.mean()),
                 "seconds": round(baseline_seconds, 1)},
    "best": {"config": best["label"], "clip_text_generated": float(best["clip_score_mean"])},
    "reference_real_photos": REFERENCE,
    "ddpm_illustration": {k_: round(v, 1) for k_, v in timings.items()},
    "configurations_run": int(len(all_results)),
    "configurations_failed": int((all_results["status"] == "error").sum()),
    "total_gpu_minutes": round(all_results["seconds"].sum() / 60, 1),
    "vram": vram(),
}
(TABLES_DIR / "milestone_2_sd15.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(TABLES_DIR / "milestone_2_sd15.json")
print(SWEEP_DIR / "all_results.csv")
print(json.dumps({**summary["baseline"], **summary["best"]}, indent=2))

### What this notebook produced

| Deliverable | Where |
|---|---|
| Explicit `prompt_embeds` integration, verified against `prompt=` | §3, §4 |
| Baseline conditional generation on the fixed 10-prompt set | §7 |
| CFG sweep (1–15) with CLIP score per setting | §8 |
| Sampler comparison across seven samplers, fixed latents | §8 |
| Noise-schedule comparison | §8 |
| DDPM illustration — 1000 steps against 20 | §8 |
| First set of generated images recorded, with the results table | §7, §9 |

The GPT-2 encoder row is in `milestone_2_encoders.ipynb`; aMUSEd and LDM-BERT are in
`milestone_2_comparison.ipynb`.

In [ ]:
del pipe, clip_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("released:", vram())